# NumPy Snippets — Exercises

Companion to the note [NumPy Snippets](NumPy%20Snippets.md).

NumPy drills built around the note's snippets: broadcasting rules, views versus copies, boolean and fancy indexing, `einsum`, vectorising loops, stable softmax/log-sum-exp, standardisation without leakage, pairwise distances and RBF kernels, least squares in both layouts, logistic-regression GD, PCA via SVD, k-means and group-by tricks. Every exercise is checked against SciPy, scikit-learn, pandas or a plain Python loop.

**18 exercises** · ✍️ By hand ×3 · 🧠 Concept ×3 · 💻 Code ×12

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from scipy.special import softmax as sp_softmax, logsumexp
from scipy.spatial.distance import cdist
from sklearn.datasets import load_iris, load_digits, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)

## Part A · Concepts and predictions

### Exercise 1 · Predict the broadcast shape
*✍️ By hand · ★☆☆*

Broadcasting aligns shapes **from the right**; two dimensions are compatible if they are equal or one of them is 1. Predict the
result shape (as a tuple) or the string `'error'` for:

(a) `(3, 1) + (4,)`  (b) `(5, 1, 4) * (3, 1)`  (c) `(2, 3) + (3,)`  (d) `(2, 3) + (2,)`  (e) `(8, 1, 6, 1) + (7, 1, 5)`

In [ ]:
shape_a = None; shape_b = None; shape_c = None; shape_d = None; shape_e = None

def _bs(s1, s2):
    try:
        return np.broadcast_shapes(s1, s2)
    except ValueError:
        return 'error'
for _name, _got, _s in [('a', shape_a, ((3, 1), (4,))), ('b', shape_b, ((5, 1, 4), (3, 1))), ('c', shape_c, ((2, 3), (3,))),
                        ('d', shape_d, ((2, 3), (2,))), ('e', shape_e, ((8, 1, 6, 1), (7, 1, 5)))]:
    _exp = _bs(*_s)
    check(f'({_name})', None if _got is None else (_got == _exp), True)

<details>
<summary><b>💡 Hint</b></summary>

Pad the shorter shape with 1s on the **left**, then compare dimension by dimension.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) `(3,1)` vs `(1,4)` → `(3, 4)`. (b) `(5,1,4)` vs `(1,3,1)` → `(5, 3, 4)`. (c) `(2, 3)`.
(d) `(2,3)` vs `(1,2)`: 3 ≠ 2 → `'error'` (to add per-row values use `b[:, None]`, shape `(2,1)`). (e) `(8,1,6,1)` vs `(1,7,1,5)` → `(8, 7, 6, 5)`.
This is how `X - X.mean(0)` (shape `(N,d)` minus `(d,)`) and `D2 = a[:, None] + b[None]` in the note work.

```python
shape_a = (3, 4); shape_b = (5, 3, 4); shape_c = (2, 3); shape_d = 'error'; shape_e = (8, 7, 6, 5)
```

</details>

### Exercise 2 · Views versus copies
*🧠 Concept · ★☆☆*

For `a = np.arange(10)`, which of these modify `a` afterwards? Explain using *views* versus *copies*.

1. `b = a[2:5]; b[0] = 99`  2. `c = a[[2, 3, 4]]; c[0] = 99`  3. `d = a[a > 5]; d[:] = 0`  4. `a[a > 5] = 0`
5. `e = a.reshape(2, 5); e[0, 0] = -1`  6. `f = a.copy(); f[0] = 7`

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Basic slicing returns a view; fancy (integer-array) and boolean indexing return copies, except when used on the left of `=`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **modifies** `a`: slices are views sharing memory. 2. No: fancy indexing copies. 3. No: boolean indexing copies.
4. **modifies**: indexed *assignment* writes into `a` in place. 5. **modifies**: `reshape` returns a view when possible.
6. No: explicit copy. Use `np.shares_memory(x, y)` to check. Unintended views are a classic source of bugs, for example
standardising a slice in place.

</details>

### Exercise 3 · Standardise with training statistics only
*🧠 Concept · ★☆☆*

The note standardises with `mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-8` and applies them to both train and test.
(a) Why not compute `mu, sd` on all data or separately on the test set? (b) What is the `1e-8` for?
(c) What does `axis=0` mean here and what would `axis=1` compute?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The test set stands in for future data you have not seen yet.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Using test statistics leaks information about the test set into preprocessing (optimistic scores); standardising the test set
with its own statistics applies a *different* transformation than the model was trained on (and breaks for a single test sample).
(b) It avoids division by zero for constant features (e.g. always-black pixels in digits).
(c) `axis=0` reduces over rows, giving one mean per feature (shape `(d,)`); `axis=1` would give one mean per sample. In sklearn
this is `StandardScaler` inside a `Pipeline` ([[scikit-learn Recipes]]).

</details>

### Exercise 4 · Why subtract the max in softmax?
*🧠 Concept · ★★☆*

Show that $\mathrm{softmax}(z - c) = \mathrm{softmax}(z)$ for any constant $c$. What goes wrong numerically for
$z = (1000, 1001)$ without the shift, and for $z=(-1000,-1001)$? Why are `axis=1, keepdims=True` needed in the note's version?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$e^{z_k-c} = e^{z_k}e^{-c}$ and the factor $e^{-c}$ cancels in the ratio. `np.exp(710)` overflows float64.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{e^{z_k-c}}{\sum_je^{z_j-c}} = \frac{e^{-c}e^{z_k}}{e^{-c}\sum_je^{z_j}}$. For $z=(1000,1001)$, `exp` overflows to `inf` and
`inf/inf = nan`; for $z=(-1000,-1001)$ both underflow to $0$ and $0/0=$ `nan`. After subtracting the max the largest exponent is
$e^0=1$, so the denominator is $\ge1$ and nothing overflows. `axis=1` takes the max/sum per sample (row), and `keepdims=True`
keeps shape `(N, 1)` so that broadcasting against `(N, K)` works row-wise.

</details>

## Part B · By hand

### Exercise 5 · Pairwise distances via the expansion trick
*✍️ By hand · ★★☆*

Show $\lVert a-b\rVert^2 = \lVert a\rVert^2+\lVert b\rVert^2-2a^\top b$, which gives the note's
`D2 = (A**2).sum(1)[:, None] + (B**2).sum(1)[None] - 2 * A @ B.T`. Compute `D2` by hand for
$A = \begin{pmatrix}0&0\\1&1\end{pmatrix}$ (rows are points) and $B = \begin{pmatrix}1&0\\2&2\end{pmatrix}$.

In [ ]:
A_pt = np.array([[0., 0.], [1., 1.]]); B_pt = np.array([[1., 0.], [2., 2.]])
D2_hand = None   # 2x2 nested list, D2[i, j] = ||A_i - B_j||^2

check('D2', D2_hand, cdist(A_pt, B_pt, 'sqeuclidean'))

<details>
<summary><b>💡 Hint</b></summary>

Row norms: $\lVert A_i\rVert^2 = (0, 2)$, $\lVert B_j\rVert^2 = (1, 8)$; $AB^\top = \begin{pmatrix}0&0\\1&4\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\lVert a-b\rVert^2 = (a-b)^\top(a-b) = a^\top a - 2a^\top b + b^\top b$.
$D2 = \begin{pmatrix}0+1-0 & 0+8-0\\ 2+1-2 & 2+8-8\end{pmatrix} = \begin{pmatrix}1&8\\1&2\end{pmatrix}$.
The trick replaces an $(N, M, d)$ intermediate with one matrix product, but can produce tiny negative values by cancellation:
clip with `np.maximum(D2, 0)` before taking square roots.

```python
A_pt = np.array([[0., 0.], [1., 1.]]); B_pt = np.array([[1., 0.], [2., 2.]])
D2_hand = [[1, 8], [1, 2]]
```

</details>

### Exercise 6 · Write the einsum strings
*✍️ By hand · ★★☆*

Write the `np.einsum` subscript string for each operation (store them as strings):

(a) trace of a square matrix `A`  (b) matrix product `A @ B`  (c) batched matrix product of `(b, n, m)` and `(b, m, p)`
(d) row-wise dot products of two `(N, d)` arrays  (e) outer product of vectors `x`, `y`  (f) the bilinear form $x^\top Ay$

In [ ]:
es_trace = None; es_matmul = None; es_batch = None; es_rowdot = None; es_outer = None; es_bilinear = None

_A = rng.normal(size=(3, 3)); _B = rng.normal(size=(3, 4)); _P = rng.normal(size=(5, 3, 4)); _Q = rng.normal(size=(5, 4, 2))
_U = rng.normal(size=(6, 3)); _V = rng.normal(size=(6, 3)); _x = rng.normal(size=3); _y = rng.normal(size=3)
_cases = [('trace', es_trace, (_A,), np.trace(_A)), ('matmul', es_matmul, (_A, _B), _A @ _B),
          ('batched matmul', es_batch, (_P, _Q), _P @ _Q), ('row-wise dot', es_rowdot, (_U, _V), (_U * _V).sum(1)),
          ('outer', es_outer, (_x, _y), np.outer(_x, _y)), ('bilinear', es_bilinear, (_x, _A, _y), _x @ _A @ _y)]
for _n, _s, _ops, _ref in _cases:
    try:
        check(_n, None if _s is None else np.einsum(_s, *_ops), _ref)
    except Exception as _e:
        print('❌', _n, 'raised', _e)

<details>
<summary><b>💡 Hint</b></summary>

Repeated indices are multiplied and, if they do not appear after `->`, summed. Example: `'ij,j->i'` is a matrix–vector product.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) `'ii->'` (b) `'ij,jk->ik'` (c) `'bij,bjk->bik'` (d) `'ij,ij->i'` (e) `'i,j->ij'` (f) `'i,ij,j->'`.
Rule of thumb: write the formula with explicit indices ($\sum_j A_{ij}B_{jk}$), list each operand's indices, and put the free indices after `->`.

```python
es_trace = 'ii->'; es_matmul = 'ij,jk->ik'; es_batch = 'bij,bjk->bik'; es_rowdot = 'ij,ij->i'; es_outer = 'i,j->ij'; es_bilinear = 'i,ij,j->'
```

</details>

## Part C · Drills

### Exercise 7 · Indexing drills
*💻 Code · ★☆☆*

With `M = np.arange(20).reshape(4, 5)` produce, **without loops**:

(a) `last_col`: the last column (shape `(4,)`); (b) `rev_rows`: the rows in reverse order; (c) `even_big`: all entries that
are even and greater than 10 (1-D); (d) `sub`: the $2\times2$ sub-matrix from rows 0, 2 and columns 1, 3; (e) `clipped`:
a copy of `R` (random normal $3\times4$) with negative values replaced by 0, leaving `R` unchanged.

In [ ]:
M = np.arange(20).reshape(4, 5)
R = rng.normal(size=(3, 4))
last_col = None; rev_rows = None; even_big = None; sub = None; clipped = None

_ML = M.tolist()
check('last column', last_col, [row[-1] for row in _ML])
check('reversed rows', rev_rows, _ML[::-1])
check('even and > 10', even_big, [v for row in _ML for v in row if v % 2 == 0 and v > 10])
check('sub-matrix', sub, [[_ML[i][j] for j in (1, 3)] for i in (0, 2)])
check('clipped copy', clipped, [[max(v, 0) for v in row] for row in R.tolist()])
check('R unchanged', None if clipped is None else bool((R < 0).any()), True)

<details>
<summary><b>💡 Hint</b></summary>

(c) combine masks with `&` and parentheses: `M[(M % 2 == 0) & (M > 10)]`. (d) `M[np.ix_([0, 2], [1, 3])]`; note that `M[[0, 2], [1, 3]]` gives only 2 elements. (e) `np.where` or `np.maximum`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`M[[0,2],[1,3]]` pairs the index arrays elementwise (giving `M[0,1]` and `M[2,3]`), whereas `np.ix_` builds the outer product of indices. `np.maximum(R, 0)` is ReLU and returns a new array.

```python
M = np.arange(20).reshape(4, 5)
R = rng.normal(size=(3, 4))
last_col = M[:, -1]
rev_rows = M[::-1]
even_big = M[(M % 2 == 0) & (M > 10)]
sub = M[np.ix_([0, 2], [1, 3])]
clipped = np.maximum(R, 0)
```

</details>

### Exercise 8 · Vectorised standardisation
*💻 Code · ★☆☆*

Implement `fit_standardizer(X_tr)` returning `(mu, sd)` and `transform(X, mu, sd)` as in the note (with the `+1e-8`). Apply them to
a train/test split of the digits data (which has constant pixels!) and compare with `StandardScaler`.

In [ ]:
def fit_standardizer(X_tr):
    return None
def transform(X, mu, sd):
    return None

X_dg, y_dg = load_digits(return_X_y=True)
Xd_tr, Xd_te = train_test_split(X_dg, test_size=0.25, random_state=0)
stats_ = fit_standardizer(Xd_tr)
Xd_te_std = None if stats_ is None else transform(Xd_te, *stats_)

_nc = Xd_tr.std(0) > 0     # compare the columns that are not constant in the training set
check('standardised test set (non-constant columns)', None if Xd_te_std is None else Xd_te_std[:, _nc], StandardScaler().fit(Xd_tr).transform(Xd_te)[:, _nc], tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

`X_tr.mean(0)` and `X_tr.std(0)` broadcast against `(N, 64)` arrays.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On non-constant columns the two agree. For pixels that are constant (always 0) in the training set, StandardScaler divides by 1 instead of $10^{-8}$. If such a pixel is non-zero in a test image, the `+1e-8` version would produce values around $10^{8}$ (in this particular split it does not happen: the printed max is about 52), a good reason to prefer the scaler's convention or to drop constant features.

```python
def fit_standardizer(X_tr):
    return X_tr.mean(0), X_tr.std(0) + 1e-8
def transform(X, mu, sd):
    return (X - mu) / sd

X_dg, y_dg = load_digits(return_X_y=True)
Xd_tr, Xd_te = train_test_split(X_dg, test_size=0.25, random_state=0)
stats_ = fit_standardizer(Xd_tr)
Xd_te_std = None if stats_ is None else transform(Xd_te, *stats_)
print('largest |value| in the standardised test set:', np.abs(Xd_te_std).max())
```

</details>

### Exercise 9 · Stable softmax and log-sum-exp
*💻 Code · ★☆☆*

Implement row-wise `softmax(Z)` and `logsumexp_rows(Z)` that survive logits like $\pm1000$. Compare with SciPy on an array that
contains huge values.

In [ ]:
def softmax(Z):
    return None
def logsumexp_rows(Z):
    return None

Z_big = np.array([[1000., 1001., 999.], [-1000., -1001., -1002.], [0., 1., 2.]])

check('softmax', softmax(Z_big), sp_softmax(Z_big, axis=1))
check('logsumexp', logsumexp_rows(Z_big), logsumexp(Z_big, axis=1))

<details>
<summary><b>💡 Hint</b></summary>

`m = Z.max(1, keepdims=True)`; `lse = m[:, 0] + np.log(np.exp(Z - m).sum(1))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both use the same shift. In practice compute `log_softmax = Z - lse[:, None]` directly rather than `np.log(softmax(Z))`, which can hit `log(0)`.

```python
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)
def logsumexp_rows(Z):
    m = Z.max(axis=1, keepdims=True)
    return m[:, 0] + np.log(np.exp(Z - m).sum(axis=1))

Z_big = np.array([[1000., 1001., 999.], [-1000., -1001., -1002.], [0., 1., 2.]])
```

</details>

### Exercise 10 · One-hot encoding in one line
*💻 Code · ★☆☆*

Write `one_hot(y, K)` returning an `(N, K)` float matrix using `np.eye` and fancy indexing, and `from_one_hot(Y)` that recovers the labels.
Then build the target matrix `T` in the note's **columns-as-samples** layout (shape `(K, N)`) for the iris labels.

In [ ]:
def one_hot(y, K):
    return None
def from_one_hot(Y):
    return None

X_ir, y_ir = load_iris(return_X_y=True)
T_cols = None   # (3, 150)

check('one-hot', one_hot(y_ir, 3), (y_ir[:, None] == np.arange(3)).astype(float))
check('round trip', None if one_hot(y_ir, 3) is None else from_one_hot(one_hot(y_ir, 3)), y_ir)
check('T shape', None if T_cols is None else T_cols.shape, (3, 150))

<details>
<summary><b>💡 Hint</b></summary>

`np.eye(K)[y]` picks row `y_i` of the identity for each sample.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`np.eye(K)[y]` is the idiom; `argmax(axis=1)` inverts it. In the column layout, `T = one_hot(y, K).T`.

```python
def one_hot(y, K):
    return np.eye(K)[y]
def from_one_hot(Y):
    return Y.argmax(axis=1)

X_ir, y_ir = load_iris(return_X_y=True)
T_cols = one_hot(y_ir, 3).T
```

</details>

### Exercise 11 · Pairwise distances, RBF kernel and kNN
*💻 Code · ★★☆*

Implement `sq_dists(A, B)` with the expansion trick (clip negatives to 0) and `rbf(A, B, gamma)`. Then write
`knn_predict(X_tr, y_tr, X_te, k)` (majority vote, no loops over test points) and compare with `KNeighborsClassifier`.

In [ ]:
def sq_dists(A, B):
    return None
def rbf(A, B, gamma):
    return None
def knn_predict(X_tr, y_tr, X_te, k=5):
    return None

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(X_ir, y_ir, test_size=0.3, random_state=1, stratify=y_ir)

check('squared distances', sq_dists(Xi_tr, Xi_te), cdist(Xi_tr, Xi_te, 'sqeuclidean'), tol=1e-8)
check('RBF kernel', rbf(Xi_tr, Xi_te, 0.5), rbf_kernel(Xi_tr, Xi_te, gamma=0.5))
check('kNN predictions', knn_predict(Xi_tr, yi_tr, Xi_te, 5), KNeighborsClassifier(5).fit(Xi_tr, yi_tr).predict(Xi_te))

<details>
<summary><b>💡 Hint 1</b></summary>

`idx = np.argsort(sq_dists(X_te, X_tr), axis=1)[:, :k]` gives the neighbours; `y_tr[idx]` has shape `(N_te, k)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Vote: `np.apply_along_axis(np.bincount, 1, y_tr[idx], minlength=K).argmax(1)`, or one-hot the neighbour labels and sum over axis 1.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The one-hot trick: `np.eye(K)[y_tr[idx]].sum(1).argmax(1)` counts votes per class with no Python loop; `argmax` breaks ties towards the smallest label. A stable sort (`kind='stable'`) breaks distance ties by index like sklearn does.

```python
def sq_dists(A, B):
    D2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None] - 2 * A @ B.T
    return np.maximum(D2, 0)
def rbf(A, B, gamma):
    return np.exp(-gamma * sq_dists(A, B))
def knn_predict(X_tr, y_tr, X_te, k=5):
    idx = np.argsort(sq_dists(X_te, X_tr), axis=1, kind='stable')[:, :k]
    K = y_tr.max() + 1
    return np.eye(K)[y_tr[idx]].sum(1).argmax(1)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(X_ir, y_ir, test_size=0.3, random_state=1, stratify=y_ir)
```

</details>

### Exercise 12 · Least squares in both layouts
*💻 Code · ★★☆*

Fit a linear least-squares classifier on iris (with a bias feature) in two ways:
(a) rows-as-samples: `W_rows = np.linalg.lstsq(Xb, Y, rcond=None)[0]` with one-hot `Y` of shape `(N, 3)`;
(b) the note's columns-as-samples formula `W = T @ X.T @ np.linalg.inv(X @ X.T)` with `X` of shape `(5, N)`.
Show `W_cols == W_rows.T`, predict with `np.argmax(W @ X, axis=0)` and store the training accuracy in `acc_ls`.

In [ ]:
Xb = np.c_[np.ones(len(X_ir)), X_ir]      # (150, 5) rows-as-samples, bias first
W_rows = None
W_cols = None
acc_ls = None

_ref = np.linalg.pinv(Xb) @ np.eye(3)[y_ir]
check('rows layout', W_rows, _ref)
check('columns layout = transpose', W_cols, _ref.T)
check('training accuracy', acc_ls, np.mean(np.argmax(Xb @ _ref, axis=1) == y_ir))

<details>
<summary><b>💡 Hint</b></summary>

In the column layout, `X = Xb.T` (shape `(5, 150)`) and `T = one_hot(y_ir, 3).T`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both give the same weights (transposed): see the derivation in [[Linear Regression]]. Least squares on one-hot targets is a quick baseline (about 85% training accuracy here) but is sensitive to outliers and does not produce probabilities, unlike [[Logistic Regression]].

```python
Xb = np.c_[np.ones(len(X_ir)), X_ir]
W_rows = np.linalg.lstsq(Xb, one_hot(y_ir, 3), rcond=None)[0]
X_c = Xb.T; T = one_hot(y_ir, 3).T
W_cols = T @ X_c.T @ np.linalg.inv(X_c @ X_c.T)
acc_ls = np.mean(np.argmax(W_cols @ X_c, axis=0) == y_ir)
print(acc_ls)
```

</details>

### Exercise 13 · Logistic regression by vectorised GD
*💻 Code · ★★☆*

On the standardised breast-cancer data, run the note's loop `p = sigmoid(X @ w); w -= lr * X.T @ (p - y) / len(y)` (add a bias
column) for 2000 steps with `lr=0.1`, recording the loss. Store the test accuracy in `acc_gd`. It should be within a few points
of sklearn's `LogisticRegression`. Make the sigmoid overflow-safe.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
sc = StandardScaler().fit(Xc_tr); Xc_tr, Xc_te = sc.transform(Xc_tr), sc.transform(Xc_te)
acc_gd = None
losses = None

_acc_sk = LogisticRegression(max_iter=5000).fit(Xc_tr, yc_tr).score(Xc_te, yc_te)
check('accuracy close to sklearn', None if acc_gd is None else abs(acc_gd - _acc_sk) <= 0.03, True)
check('loss decreased', None if losses is None else losses[-1] < losses[0] / 5, True)

<details>
<summary><b>💡 Hint</b></summary>

Overflow-safe sigmoid: `scipy.special.expit`, or `np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))`. Loss: `np.mean(np.logaddexp(0, z) - y * z)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The update is the cross-entropy gradient $X^\top(\sigma(Xw)-y)/N$ ([[Matrix Calculus Cheatsheet]]). Standardisation is what makes `lr=0.1` work; without it the Hessian is badly conditioned ([[Gradient Descent]]).

```python
from scipy.special import expit
Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
sc = StandardScaler().fit(Xc_tr); Xc_tr, Xc_te = sc.transform(Xc_tr), sc.transform(Xc_te)
Xtr_b = np.c_[np.ones(len(Xc_tr)), Xc_tr]; Xte_b = np.c_[np.ones(len(Xc_te)), Xc_te]
w = np.zeros(Xtr_b.shape[1]); losses = []
for _ in range(2000):
    z = Xtr_b @ w
    losses.append(np.mean(np.logaddexp(0, z) - yc_tr * z))
    w -= 0.1 * Xtr_b.T @ (expit(z) - yc_tr) / len(yc_tr)
acc_gd = np.mean((Xte_b @ w > 0) == yc_te)
print(acc_gd)
```

</details>

### Exercise 14 · PCA via SVD
*💻 Code · ★★☆*

Implement `pca_svd(X, k)` returning `(Z, components, explained_variance_ratio)` as in the note (centre, SVD, project onto
`Vt[:k]`). Apply it to the digits data with $k=10$ and compare with `sklearn.decomposition.PCA` (signs may differ).

In [ ]:
def pca_svd(X, k):
    return None

res_pca = pca_svd(X_dg, 10)

_p = PCA(10, svd_solver='full').fit(X_dg)
check('|scores|', None if res_pca is None else np.abs(res_pca[0]), np.abs(_p.transform(X_dg)), tol=1e-6)
check('explained variance ratio', None if res_pca is None else res_pca[2], _p.explained_variance_ratio_)

<details>
<summary><b>💡 Hint</b></summary>

Explained variance ratio $=\sigma_i^2/\sum_j\sigma_j^2$ using **all** singular values.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The principal axes are the rows of `Vt`, the scores `Xc @ Vt[:k].T = U[:, :k] * S[:k]`. The sign of each singular vector is arbitrary, so compare absolute values (sklearn applies a deterministic sign flip). See [[PCA]] and [[Linear Algebra for ML]].

```python
def pca_svd(X, k):
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:k].T, Vt[:k], (S[:k] ** 2) / np.sum(S ** 2)

res_pca = pca_svd(X_dg, 10)
```

</details>

### Exercise 15 · k-means without the empty-cluster bug
*💻 Code · ★★☆*

The note's k-means crashes (mean of an empty slice gives `nan`) if a cluster loses all its points. Implement
`kmeans(X, C0, iters=100)` that (i) assigns with broadcasting, (ii) keeps the old centre if a cluster is empty, and (iii) stops
when assignments no longer change. Starting from the given centres on iris, compare with `KMeans(init=C0, n_init=1)`.

In [ ]:
def kmeans(X, C0, iters=100):
    # TODO: return (centres, labels)
    return None, None

C0 = X_ir[[0, 50, 100]].astype(float)
C_km, a_km = kmeans(X_ir, C0)

_km = KMeans(3, init=C0, n_init=1, tol=0, algorithm='lloyd').fit(X_ir)
check('centres', C_km, _km.cluster_centers_, tol=1e-8)
check('labels', a_km, _km.labels_)

<details>
<summary><b>💡 Hint</b></summary>

`a = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)` has an `(N, k, d)` intermediate; fine for small data, use `sq_dists` for big data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With a deterministic initialisation both implementations run the same Lloyd iterations. Real implementations use k-means++ seeding and several restarts (`n_init`) because the result depends on the start ([[Clustering]]).

```python
def kmeans(X, C0, iters=100):
    C = C0.copy(); a = None
    for _ in range(iters):
        a_new = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)
        if a is not None and np.array_equal(a, a_new):
            break
        a = a_new
        for j in range(len(C)):
            if np.any(a == j):
                C[j] = X[a == j].mean(0)
    return C, a

C0 = X_ir[[0, 50, 100]].astype(float)
C_km, a_km = kmeans(X_ir, C0)
```

</details>

### Exercise 16 · einsum in practice: Mahalanobis distances
*💻 Code · ★★☆*

For points `P` (shape `(N, d)`), mean `mu` and covariance `S`, compute all Mahalanobis distances
$\sqrt{(x_n-\mu)^\top S^{-1}(x_n-\mu)}$ in one `np.einsum` call (no loop), and a batch of quadratic forms for a stack of
matrices `Ms` (shape `(B, d, d)`) and vectors `vs` (shape `(B, d)`): $v_b^\top M_bv_b$.

In [ ]:
P = rng.normal(size=(200, 3)); mu_p = P.mean(0); S_p = np.cov(P, rowvar=False)
Ms = rng.normal(size=(10, 3, 3)); vs = rng.normal(size=(10, 3))
maha = None
quad_forms = None

check('Mahalanobis', maha, cdist(P, mu_p[None], 'mahalanobis', VI=np.linalg.inv(S_p))[:, 0])
check('batched quadratic forms', quad_forms, [v @ M @ v for M, v in zip(Ms, vs)])

<details>
<summary><b>💡 Hint</b></summary>

`D = P - mu`; `np.einsum('ni,ij,nj->n', D, Sinv, D)`. For the batch: `'bi,bij,bj->b'`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

einsum avoids building an $(N, N)$ matrix (`D @ Sinv @ D.T` would compute all cross terms and then take the diagonal), so it is $O(Nd^2)$ instead of $O(N^2d)$.

```python
P = rng.normal(size=(200, 3)); mu_p = P.mean(0); S_p = np.cov(P, rowvar=False)
Ms = rng.normal(size=(10, 3, 3)); vs = rng.normal(size=(10, 3))
D = P - mu_p
maha = np.sqrt(np.einsum('ni,ij,nj->n', D, np.linalg.inv(S_p), D))
quad_forms = np.einsum('bi,bij,bj->b', vs, Ms, vs)
```

</details>

### Exercise 17 · Moving averages with cumsum
*💻 Code · ★★☆*

Compute the moving average of window $w=7$ of a noisy signal in three ways and store the results: (a) with `np.cumsum`
(O(N), no loop), (b) with `np.lib.stride_tricks.sliding_window_view(x, w).mean(1)`. Both must equal
`np.convolve(x, np.ones(w)/w, mode='valid')`.

In [ ]:
x_sig = np.sin(np.linspace(0, 6 * np.pi, 300)) + 0.3 * rng.normal(size=300)
ma_cumsum = None
ma_windows = None

_ref = np.array([x_sig[i:i + 7].mean() for i in range(len(x_sig) - 6)])
check('cumsum moving average', ma_cumsum, _ref)
check('sliding-window moving average', ma_windows, _ref)

<details>
<summary><b>💡 Hint</b></summary>

With `c = np.cumsum(np.r_[0, x])`, the sum of `x[i:i+w]` is `c[i+w] - c[i]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The cumsum trick computes all window sums in $O(N)$; `sliding_window_view` is a zero-copy strided view of shape `(N-w+1, w)`, convenient but $O(Nw)$ when reduced. Cumsum can accumulate round-off for very long float32 signals.

```python
x_sig = np.sin(np.linspace(0, 6 * np.pi, 300)) + 0.3 * rng.normal(size=300)
c = np.cumsum(np.r_[0., x_sig])
ma_cumsum = (c[7:] - c[:-7]) / 7
ma_windows = np.lib.stride_tricks.sliding_window_view(x_sig, 7).mean(1)
```

</details>

### Exercise 18 · Group-by without pandas
*💻 Code · ★★★*

Compute the per-class mean image of the digits data (shape `(10, 64)`) and the per-class count **without a Python loop over
classes**, using `np.add.at` or `np.bincount`. Compare with `pandas.DataFrame.groupby`. Then use the class means as a nearest-centroid
classifier and store its training accuracy in `acc_centroid`.

In [ ]:
class_means = None
class_counts = None
acc_centroid = None

_df = pd.DataFrame(X_dg); _df['y'] = y_dg
check('class means', class_means, _df.groupby('y').mean().values)
check('class counts', class_counts, _df.groupby('y').size().values)
check('nearest-centroid accuracy', acc_centroid, np.mean(cdist(X_dg, _df.groupby('y').mean().values).argmin(1) == y_dg))

<details>
<summary><b>💡 Hint</b></summary>

`S = np.zeros((10, 64)); np.add.at(S, y_dg, X_dg)` sums rows into their class; counts are `np.bincount(y_dg)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`np.add.at` is unbuffered, so repeated indices accumulate (unlike `S[y] += X`, which would keep only one row per class). An alternative is the one-hot matrix product `one_hot(y).T @ X`. Nearest centroid reaches about 90% on digits, a strong baseline for one line of maths.

```python
S = np.zeros((10, X_dg.shape[1])); np.add.at(S, y_dg, X_dg)
class_counts = np.bincount(y_dg)
class_means = S / class_counts[:, None]
acc_centroid = np.mean(sq_dists(X_dg, class_means).argmin(1) == y_dg)
print(acc_centroid)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [NumPy Snippets](NumPy%20Snippets.md).*